# 01 - Generar entregables del challenge

Notebook operativo para reconstruir el proyecto desde cero: ingesta, dataset `comercial`, dashboard HTML y validaciones.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)


## 1. Validar configuracion

Valida que existen credenciales en `.env` sin mostrar la contrasena.


In [ ]:
from src.utils.config_loader import load_db_config, get_jdbc_url

db_config = load_db_config(require_credentials=True)
print("Host:", db_config["host"])
print("Database:", db_config["database"])
print("User configured:", bool(db_config["user"]))
print("Password configured:", bool(db_config["password"]))
print("JDBC URL:", get_jdbc_url(db_config))


## 2. Crear SparkSession


In [ ]:
from src.utils.spark_session import create_spark_session

spark = create_spark_session("challenge_entregables")
print("Spark version:", spark.version)


## 3. Ingerir tablas fuente

Ingiere `customers`, `products`, `sales` y `shops` hacia `data/raw/`.


In [ ]:
from src.ingest.extract_tables import ingest_tables

raw_counts = ingest_tables(spark=spark)
raw_counts


## 4. Construir dataset comercial

Genera `data/analytics/comercial/` en formato Parquet.


In [ ]:
from src.transform.build_comercial import build_comercial_table

comercial_rows = build_comercial_table(spark=spark)
print("Rows comercial:", comercial_rows)


## 5. Validar Parquet comercial


In [ ]:
comercial_path = PROJECT_ROOT / "data" / "analytics" / "comercial"
comercial_df = spark.read.parquet(str(comercial_path))
print("Rows:", comercial_df.count())
comercial_df.select("invoice_id", "sale_date", "shop_name", "product_name", "customer_status", "subtotal").show(10, truncate=False)


## 6. Generar dashboard HTML

Genera `dashboard/index.html` con KPI y 4 graficos.


In [ ]:
from src.export.dashboard_data import generate_dashboard

payload = generate_dashboard()
print("Rows dashboard:", payload["metadata"]["row_count"])
print("Total vendido:", payload["kpis"]["total_sold"])
print("Archivo:", PROJECT_ROOT / "dashboard" / "index.html")


## 7. Validar componentes del dashboard


In [ ]:
html = (PROJECT_ROOT / "dashboard" / "index.html").read_text(encoding="utf-8")
required = ["Total vendido", "dailySalesChart", "shopPieChart", "productBarChart", "customerStatusChart"]
for marker in required:
    print(marker, marker in html)


## 8. Cerrar Spark


In [ ]:
spark.stop()
print("Spark stopped")
